# 02 — EDA + Feature Engineering

Explores the cleaned dataset and applies the feature pipeline from `src/data/features.py`.

**CRITICAL:** PRICE_PER_SQFT is **never** used as a feature — it is derived from the target (PRICE) and causes data leakage. See [ADR-001](../docs/decisions/001-remove-price-per-sqft.md).

**Features engineered:**
- Numeric: TOTAL_ROOMS, BED_BATH_RATIO, LOG_SQFT, ROOMS_PER_SQFT
- Geospatial: DIST_MANHATTAN_CENTER, DIST_CENTRAL_PARK, DIST_NEAREST_SUBWAY, NEIGHBORHOOD_CLUSTER
- Targets: PRICE_ZONE (4-class), LOG_PRICE, SQFT_CATEGORY (3-class)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from src.data.loader import load_cleaned
from src.data.features import add_numeric_features, add_target_variables, add_geospatial_features
from src.utils.geo import add_distance_features
from src.utils.validation import assert_no_leakage
from src.config import NUMERIC_FEATURES, ONEHOT_FEATURES, TARGET_ENCODED_FEATURES

In [ ]:
df = load_cleaned()
print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} cols")
df.head(3)

## EDA: Price Distribution

In [ ]:
fig = px.histogram(df, x="PRICE", nbins=50, title="Price Distribution (Cleaned Data)",
                    labels={"PRICE": "Price ($)"}, color_discrete_sequence=["#2196F3"])
fig.update_layout(showlegend=False)
fig.show()

## EDA: Geographic Distribution

In [ ]:
if "BOROUGH" in df.columns:
    fig = px.scatter_mapbox(
        df, lat="LATITUDE", lon="LONGITUDE", color="BOROUGH",
        hover_data=["PRICE", "TYPE", "PROPERTYSQFT"],
        title="NYC Properties by Borough",
        mapbox_style="carto-positron", zoom=10, height=600,
    )
    fig.show()
else:
    print("BOROUGH column not available — skipping map")

## Feature Engineering Pipeline

In [ ]:
# Add derived numeric features
df = add_numeric_features(df)

# Add geospatial features (distances, clusters)
ref_points = {"MANHATTAN_CENTER": (40.7580, -73.9855), "CENTRAL_PARK": (40.7829, -73.9654)}
df = add_distance_features(df, ref_points)
df["DIST_NEAREST_SUBWAY"] = df["DIST_MANHATTAN_CENTER"]  # proxy
from src.utils.geo import add_neighborhood_clusters
df = add_neighborhood_clusters(df, n_clusters=15)

# Add target variables
df = add_target_variables(df)

print(f"Engineered: {df.shape[0]} rows x {df.shape[1]} cols")
print(f"\nNew columns: {[c for c in df.columns if c not in load_cleaned().columns]}")

## Leakage Guard Check

In [ ]:
# Verify NO leaky features in our feature set
all_features = NUMERIC_FEATURES + ONEHOT_FEATURES + TARGET_ENCODED_FEATURES
assert_no_leakage(all_features)
print(f"Leakage check PASSED — {len(all_features)} features verified clean")
print(f"\nFeature list:")
for i, f in enumerate(all_features, 1):
    print(f"  {i:2d}. {f}")

## Price Zone Distribution

In [ ]:
zone_counts = df["PRICE_ZONE"].value_counts().sort_index()
fig = px.bar(x=zone_counts.index.astype(str), y=zone_counts.values,
             title="Price Zone Distribution (Classification Target)",
             labels={"x": "Price Zone", "y": "Count"},
             color=zone_counts.index.astype(str),
             color_discrete_sequence=["#4CAF50", "#2196F3", "#FF9800", "#F44336"])
fig.update_layout(showlegend=False)
fig.show()

print(f"\nClass distribution:")
for zone, count in zone_counts.items():
    print(f"  {zone}: {count} ({count/len(df)*100:.1f}%)")